## အင်ဘက်ဒင်းများ

ကြှနျုပျတို့၏ ယခင် နမူနာတွင်၊ ကျနော်တို့သည် အမြင့်-မပြည့် အရှည် `vocab_size` ရှိသော bag-of-words vectors များအပေါ် လုပ်ဆောင်ခဲ့ပြီး၊ ကျနော်တို့သည် အနိမ့်-အရွယ်အစား ရှိသော အနေအထားပြသည့် vectors များကို စပျင်း-တစ်ချက်တည်း အပြသပြုပုံသို့ တိတိကျကျ ပြောင်းလဲခဲ့ကြသည်။ ဒီတစ်ချက်တည်း ဖြင့် ဖော်ပြထားသော ပြသည် memory-ထိရောက်မှု မရှိဘဲဖြစ်ပြီး၊ စကားလုံးတိုင်းကို တစ်ခုချင်းစီ ကိုယ်ပိုင်အနေဖြင့် ဆက်ဆံသည်၊ ဥပမာအားဖြင့် တစ်ချက်တည်း ကုဒ်ထားသော vectors များသည် စကားလုံးများ ကြားရှိ သဘောထားယှဉ်တူမှု မပြသနိုင်ပါ။

ဒီယူနစ်တွင် ကျနော်တို့သည် **News AG** ဒေတာစုစည်းမှုကို ဆက်လက်လေ့လာသွားမည်ဖြစ်သည်။ စတင်ရန်အတွက် အချက်အလက်များကို စတိုး၍ ယခင် notebook မှ ပြောပြချက်အချို့ကို ရယူလိုက်ပါစို့။


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## embedding ဆိုတာ ဘာလဲ?

**embedding** ဆိုတာက စကားလုံးကို အနည်းဆုံး အတိုင်းအတာနဲ့ ပြည့်နှက်သော vector များဖြင့် ကိုယ်စားပြုခြင်းဖြစ်ပြီး၊ အဲဒါက စကားလုံးတစ်လုံးရဲ့ အဓိပ္ပာယ်ကို တစ်သွယ်တည်း ဖော်ပြပေးနိုင်ပါတယ်။ နောက်ပိုင်းမှာ မှန်ကန်သော word embedding များကို မည်ကဲ့သို့တည်ဆောက်မလဲ ဆိုတာကို ဆွေးနွေးပါမယ်၊ ဒါပေမယ့် ပထမဦးစွာ embedding ကို စကားလုံး vector တစ်ခုရဲ့ အတိုင်းအတာလျော့ဖြစ်စေမှုလို့ ထင်ကြည့်ကြပါစို့။

ထို့ကြောင့် embedding အလွှာက စကားလုံးတစ်လုံးကို input အဖြစ်ယူပြီး သတ်မှတ်ထားသော `embedding_size` ရဲ့ output vector တစ်ခု ထုတ်ပေးပါလိမ့်မယ်။ အနေနဲ့တော့ `Linear` အလွှာနဲ့ အလွန်ဆင်တူပါတယ်၊ သို့သော် one-hot encoded vector တစ်ခုကို ယူခြင်းမဟုတ်ဘဲ စကားလုံး အမှတ်ကို input အဖြစ် ခံနိုင်မှာဖြစ်ပါတယ်။

embedding အလွှာကို ကျွန်တော်တို့ network ရဲ့ ပထမအလွှာအဖြစ် အသုံးပြုခြင်းဖြင့် ယခင် bag-of-words မှ **embedding bag** ပုံစံသို့ ပြောင်းလဲနိုင်ပြီး၊ အဲဒီမှာ စာသားထဲရှိ စကားလုံးအားလုံးကို သတ်မှတ်ချက် embedding vector များသို့ ပြောင်း၍ ဤ vector များအပေါ်မှာ `sum`, `average` သို့မဟုတ် `max` ကဲ့သို့ aggregate function တစ်ခုကိုတွက်ချက်နိုင်ပါသည်။

![ စီးရီးတစ်ခုတွင် စကားလုံးငါးလုံးအတွက် embedding classifier ကို ပြသသည့် ပုံ။](../../../../../translated_images/my/embedding-classifier-example.b77f021a7ee67eee.webp)

ကျွန်တော်တို့ရဲ့ classifier neural network က embedding အလွှာနဲ့ စတင်ပြီး aggregation layer နဲ့ ထိပ်မှာ linear classifier တစ်ခုပါဝင်ပါလိမ့်မယ်။


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### အကြောင်းအရာစဉ်အရွယ်အစား ကိစ္စဖြေရှင်းခြင်း

ဒီဒီဇိုင်းစနစ်ရဲ့ ရလဒ်အနေဖြင့်၊ ကျွန်တော်တို့၏ နက်ဝပ်ခ်သို့ မီနီပက်ချ်များကို တစ်စိတ်တစ်ပိုင်းနည်းလမ်းဖြင့် ဖန်တီးရမှာဖြစ်သည်။ ယခင်ယူနစ်မှာ bag-of-words ကို အသုံးပြုတဲ့အခါ၊ မီနီပက်ချ်အတွင်းရှိ BoW တန်ဆာများအားလုံးသည် အသုံးပြုသော စာသားစဉ်၏ လက်တွေ့အရှည်မရွေးအလိုက် `vocab_size` အရွယ်အစား တူညီခဲ့သည်။ စကားလုံးတွဲချိတ်ပုံသို့ ရွှေ့ပြောင်းပြီးနောက်မှာ အဲဒီစာသားနမူနာ တစ်ခုချင်းစီမှာ စကားလုံးအရေအတွက် မတူညီသွားပြီး၊ နမူနာအများအပြားကို မီနီပက်ချ်အဖြစ်ပေါင်းစပ်ရာတွင် ချိတ်ဆွဲမှုကြောင့် padding ကို အသုံးပြုရမည်ဖြစ်သည်။

၎င်းကို datasource သို့ `collate_fn` ဖန်ရှင်ပေးခြင်းနည်းလမ်းတူ အသုံးပြု၍ ပြုလုပ်နိုင်သည်။


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### အင်ဘက်ဒင်း တွဲဖက်ခွဲခြားသူ သင်ကြားခြင်း

အခုတော့ ကျွန်တော်တို့က ယခင်ယူနစ်မှာ သတ်မှတ်ထားတဲ့ သင်ကြားမှု function ကို အသုံးပြုပြီး မော်ဒယ်ကို သင်ကြားနိုင်ပြီဖြစ်ပါတယ်။ 


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **မှတ်ချက်** - ကျွန်ုပ်တို့သည် အချိန်ရောဂါအတွက် 25k မှတ်တမ်းသာ သင်ကြားနေခြင်းဖြစ်ပြီး (အပြည့်အဝ epoch တစ်ခုထက်နည်းသည်) သို့သော် သင်သည် သင်ကြားမှုကို ဆက်လက်လုပ်ဆောင်နိုင်ပြီး၊ epoch များအတွက် သင်ကြားရန် function တစ်ခုရေးသား၍၊ အတတ်ပညာ သင်ယူမှုနှုန်း parameter ကို စမ်းသပ်ကာ တိကျမှန်ကန်မှု များအထိ ရောက်ရှိကာ ရနိုင်ပါသည်။ သင့်အား 90% အနီးဆုံသော တိကျမှန်ကန်မှုကို ရောက်ရှိနိုင်အောင် ဖြစ်စေပါသည်။


### EmbeddingBag ကွက်နှင့် အမျိုးမျိုးအရှည်ရှိ သင်္ချာလိုက်ကိုယ်စားပြုခြင်း

အရင်ဆုံး စိတ်ဖွဲ့တည်နေသောဗိသုကာတွင် ကွဲပြားသောသင်္ချာလိုက်အားလုံးကို minibatch ထဲသို့ထည့်သွင်းနိုင်ရန်အတွက် ပမာဏတူအရှည်ရှိအောင် padding လုပ်ရန်လိုအပ်ခဲ့သည်။ ဒါက အမျိုးမျိုးအရှည်ရှိ သင်္ချာလိုက်ကို ကိုယ်စားပြုဖို့အတွက် အထိရောက်ဆုံးနည်းလမ်း မဟုတ်ပါ။ အခြားနည်းလမ်းမှာ ကွက်တစ်ခုထဲတွင်သင်္ချာလိုက်အားလုံး၏ offset များကိုသိမ်းဆည်းသည့် **offset** ဗက်တာကို အသုံးပြုခြင်းဖြစ်သည်။

![Image showing an offset sequence representation](../../../../../translated_images/my/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **မှတ်ချက်**: အထက်အစားပုံတွင် သင်္ချာလိုက်ကို အက္ခရာများဖြင့် ပြသထားသော်လည်း၊ ဥပမာတွင် စကားလုံးအလိုက် သင်္ချာလိုက်များဖြင့် အလုပ်လုပ်နေပါသည်။ သင်္ချာလိုက်များကို offset ဗက်တာဖြင့် ကိုယ်စားပြုခြင်း၏ အခြေခံသဘောတရားက နည်းတူသေးသည်။

offset ကိုယ်စားပြုမှုနဲ့ အလုပ်လုပ်ဖို့ [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) ကွက်ကို အသုံးပြုပါတယ်။ ဒါဟာ `Embedding` နဲ့ဆင်တူပေမယ့် များအနေဖြင့် content vector နဲ့ offset vector ကို input အနေနဲ့ယူပြီး နောက်တွင် `mean`၊ `sum` ဒါမှမဟုတ် `max` ဖြစ်နိုင်တဲ့ averaging layer ကိုပါ ထည့်သွင်းထားပါတယ်။

ဒီမှာ `EmbeddingBag` ကို အသုံးပြုထားတဲ့ ပြင်ဆင်ထားသော ကွန်ယက်ပါ:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

သင်ကြားရေးအတွက် ဒေတာသို្ស့ စီစဉ်ရန်၊ offset vector ကိုပြင်ဆင်သွားမည့် ပြောင်းလဲမှုလုပ်ဆောင်ချက်တစ်ခု ပံ့ပိုးပေးရမည်။ 


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

သတိပြုပါ၊ ယခင်နမူနာများအားလုံးနှင့် မတူဘဲကျွန်ုပ်တို့၏ကွန်ရက်သည် ယခုတွင် မတူညီသောအရွယ်အစားရှိသော parameters နှစ်ခုကိုလက်ခံသည်၊ data vector နှင့် offset vector တို့ဖြစ်သည်။ ထို့နောက်၊ ကျွန်ုပ်တို့၏ data loader ကလည်း ၂ များမဟုတ်ဘဲ ၃ များဖြင့်ပံ့ပိုးပေးသည်၊ text နှင့် offset vectors နှစ်ခုလုံးကို features အဖြစ်ပေးထားသည်။ ထို့ကြောင့်၊ ကျွန်ုပ်တို့ training function ကို သေးငယ်စွာ အပြင်ဖျံ့ဖြိုးရန် လိုအပ်သည်။


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## အသက္သေဘာအဓိပ္ပာယ်ဖော်ထုတ်ချက်များ: Word2Vec

ကျွန်ုပ်တို့၏လွန်ခဲ့သောဥပမာတွင် မော်ဒယ် embedding အလွှာသည် စကားလုံးများကိုဗက်တာပုံသဏ္ဍာန်သို့မြေပုံပြုလုပ်ရာလေ့လာသော်လည်း၊ ဤပုံသဏ္ဍာန်သည် အဓိပ္ပာယ်ဆိုင်ရာရှုမြင်မှုမရှိခဲ့ပါ။ အမျိုးအမည်တူ သို့မဟုတ် သက်ဆိုင်ရာ စကားလုံးများသည် vector အကွာအဝေး (ဥပမာ၊ euclidian distance) အရ နီးကပ်သော vector များနှင့် ကိုက်ညီရန် ဦးတည်ချက်ထားပြီး ဤသို့သောဗက်တာပုံသဏ္ဍာန်ကိုသင်ယူနိုင်ရင်ကောင်းမည်ဖြစ်သည်။

အဲဒီအတွက် ကျွန်ုပ်တို့ သတ်မှတ်ထားသော စာသားစုစည်းထဲတွင် embedding မော်ဒယ်ကို pre-train လုပ်ရပါမည်။ semantic embedding များကို သင်ကြားရန် ပထမဆုံးနည်းလမ်းများထဲမှတစ်ခုမှာ [Word2Vec](https://en.wikipedia.org/wiki/Word2vec) ဟူ၍ခေါ်သည်။ ၎င်းသည် စကားလုံးများ၏ဖြန့်ဖြူးထားသော ကိုယ်စားပြုချက် အတွက် အသုံးပြုသော နည်းပညာနှစ်မျိုးအခြေခံထားသည်။

 - **Continuous bag-of-words** (CBoW) — ဤနည်းပညာတွင် မော်ဒယ်ကိုပုံနှိပ်လျက်ရှိသော စာလုံးမှ ပြေးပစ်မှု ကြိုတင်ခန့်မှန်းရန် သင်ကြားသည်။ ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$ ပေးထားလျှင် မော်ဒယ်၏ရည်မှန်းချက်မှာ $(W_{-2},W_{-1},W_1,W_2)$ မှ $W_0$ ကို ခန့်မှန်းခြင်းဖြစ်သည်။
 - **Continuous skip-gram** သည် CBoW ၏အလားတူနည်းဖြစ်သည်။ မော်ဒယ်သည် လက်ရှိစကားလုံးကို ခန့်မှန်းရန် ပတ်ဝန်းကျင်မှ စကားလုံးများကို အသုံးပြုသည်။

CBoW သည် အမြန်လုပ်ဆောင်နိုင်ပြီး skip-gram သည် နှေးကွေးသော်လည်း မကြာဖြစ်သော စကားလုံးများကို ပိုမိုတိကျစွာ ကိုယ်စားပြုနိုင်သည်။

![Image showing both CBoW and Skip-Gram algorithms to convert words to vectors.](../../../../../translated_images/my/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

Google News dataset အတွက် pre-trained ဖြစ်သော word2vec embedding ကို စမ်းသပ်ရန် **gensim** စာကြည့်တိုက်ကို အသုံးပြုနိုင်ပါသည်။ အောက်တွင် 'neural' နှင့် အလားတူသော စကားလုံးများကိုတွေ့နိုင်ပါသည်။

> **နိဒါန်း:** စကားလုံးဗက်တာများကိုပထမဆုံးဖန်တီးသောအခါ ဒေါင်းလုပ်လုပ်ခြင်းအချိန်အနည်းငယ်ယူနိုင်သည်!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


ဗေါ်စကားမှ စကားလုံး embeddings များကိုလည်း သတ်မှတ်နိုင်ပြီး၊ စိစစ်ခွဲခြားမှု မော်ဒယ် သင်ကြားမှုအတွက် အသုံးပြုနိုင်သည် (ရှင်းလင်းမှုအတွက် ဗက်တာ၏ ပထမ ၂၀ ကွန်ပိုနင့်သာ ပြသထားသည်)။


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

စာလုံးများရဲ့ အဓိပ္ပာယ်ဗေဒဆိုင်ရာ တင်စားမှုတွေအကြောင်းမှာ အလွန်ကောင်းတာက နေရာဗက်တာကို သုံးပြီး အဓိပ္ပာယ်တွေကို ပြောင်းလဲနိုင်တယ်ဆိုတာပါ။ ဥပမာ၊ မြေပြင်သမား *king* နဲ့ မိန်းကလေး *woman* တို့နဲ့ နီးကပ်စွာရှိပြီး၊ ယောက်ျား *man* ဆီက နောက်ကွယ်ဝေးတဲ့ စကားလုံးတစ်လုံးကို ရှာဖွေဖို့ မေးနိုင်ပါတယ်။


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

CBoW နှင့် Skip-Grams နှစ်မျိုးလုံးသည် “ခန့်မှန်းချက်” အမျိုးအစား word embeddings ဖြစ်ပြီး ဒေသန္တရ context များကိုသာ တွက်ချက်သုံးစွဲသည်။ Word2Vec သည် ကမ္ဘာလုံးဆိုင်ရာ context ကို အသုံးပြုခြင်း မရှိပါ။ 

**FastText** သည် Word2Vec ကို အခြေခံပြီး စကားလုံးတိုင်းနှင့် စကားလုံးထဲရှိ character n-grams များအတွက် vector ကို သင်ယူသည်။ ပြီးရင် တစ်ကိုယ်တော် vector ကို training အဆင့်တိုင်းမှာ တစ်ခုတည်း အနေနဲ့ ပေါင်းစပ်သည်။ ဒါက pre-training အတွက် အပိုတွက်ချက်မှုများ ထည့်သွင်းပေးသော်လည်း စကားလုံး embeddings သည် sub-word များအကြောင်းကို ကူးစက်ဖော်ပြနိုင်စေသည်။ 

နောက်ထပ်နည်းလမ်းတစ်ခုမှာ **GloVe** ဖြစ်ပြီး co-occurrence matrix အယူအဆကို အသုံးပြုကာ neural နည်းလမ်းများဖြင့် co-occurrence matrix ကို ပိုမိုဖော်ပြနိုင်ပြီး နောက်ကွယ် nonlinear ရှိသော word vectors များအဖြစ် ခွဲခြမ်းစိတ်ဖြာသည်။

gensim သည် သေချာစွာ အမျိုးမျိုးသော word embedding မော်ဒယ်များကို ထောက်ခံသောကြောင့် ပုံပြင်ကို FastText နှင့် GloVe တို့သို့ ပြောင်းလဲ၍ စမ်းသပ်နိုင်ပါသည်။


## PyTorch တွင် ကြိုပြီးလေ့လာထားသော Embeddings များ အသုံးပြုခြင်း

ကျွန်ုပ်တို့အထက်ပါဥပမာကို ပြင်ဆင်၍ embedding layer တွင် semantical embeddings များဖြစ်သော Word2Vec ကဲ့သို့သော အရာများဖြင့် matrix ကို ကြိုတင်ဖြည့်စွက်နိုင်သည်။ ကြိုပြီးလေ့လာထားသော embedding ၏ vocabulary များနှင့် ကျွန်ုပ်တို့၏ စာသား ကုဒ်စုသည် မကိုက်ညီနိုင်မှုဖြစ်နိုင်သည်ကို ဂရုစိုက်ရမည်၊ ထို့ကြောင့် စကားလုံးဆုံးရှုံးနေသော နေရာများကို random တန်ဖိုးများဖြင့် သိပ်သည်းဆ နည်းဖြင့်စတင်မည်ဖြစ်သည်။


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


ယခုကျွန်ုပ်တို့၏ မော်ဒယ်ကို များစွာ လေ့ကျင့်ကြရအောင်။ မော်ဒယ်ကို လေ့ကျင့်ရန် ကြာချိန်သည် ယခင် ဥပမာထက် များစွာ ကြီးမားသော embedding အလွှာအရွယ်အစားကြောင့်၊ မော်ဒယ်ပါပရာမီတာ အရေအတွက် များပြားထားခြင်းကြောင့် ဖြစ်သည်။ ထို့ကြောင့်လည်း မော်ဒယ်၏ overfitting ဖြစ်ခြင်းကို ရှောင်ကြဉ်လိုပါက များစွာသော ဥပမာပေါ်တွင် မော်ဒယ်အား လေ့ကျင့်ရနိုင်ပါသည်။


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

ကျွန်ုပ်တို့အတွက် တိကျမှုမှာ များပြားအောင် တိုးတက်မှုကြီးမဟုတ်ပါ၊ ၎င်းသည် ထင်ရှားစွာ ကွဲပြားသော အသုံးအနှုန်းများရှိခြင်းကြောင့် ဖြစ်နိုင်သည်။ 
ကွဲပြားသော အသုံးအနှုန်းများပြဿနာကို ဖြေရှင်းရန်အတွက် အောက်ဖော်ပြပါ ဖြေရှင်းနည်းများထဲမှ တစ်ခုကို သုံးနိုင်သည်။
* ကျွန်ုပ်တို့အသုံးအနှုန်းပေါ်တွင် word2vec မော်ဒယ်ကို ထပ်မံလေ့ကျင့်ခြင်း
* မှီခိုပြီးလေ့ကျင့်ထားသော word2vec မော်ဒယ်မှ အသုံးအနှုန်းဖြင့် ကျွန်ုပ်တို့ဒေတာစုံကို ဒေါင်းလုပ်လုပ်ခြင်း။ ဒေတာစုံဒေါင်းလုပ်လုပ်ရာတွင် အသုံးအနှုန်းကို ဖော်ပြနိုင်သည်။

နောက်ဆုံးနည်းလမ်းသည် ပိုလွယ်ကူသလို့ မြင်ရသည်၊ အထူးသဖြင့် PyTorch `torchtext` ဖရိမ်းဝတ်တွင် အင်ဘက်ဒ်မှုများကို အထောက်အပံ့ထားသောကြောင့် ဖြစ်သည်။ ဥပမာအားဖြင့် GloVe အခြေခံအသုံးအနှုန်းကို အောက်ပါအတိုင်း ဖန်တီးနိုင်သည်။


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Loaded vocabulary has the following basic operations:
* `vocab.stoi` dictionary allows us to convert word into its dictionary index
* `vocab.itos` does the opposite - converts number into word
* `vocab.vectors` is the array of embedding vectors, so to get the embedding of a word `s` we need to use `vocab.vectors[vocab.stoi[s]]`

Here is the example of manipulating embeddings to demonstrate the equation **kind-man+woman = queen** (I had to tweak the coefficient a bit to make it work):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

ထို embeddings များကို အသုံးပြု၍ classifier ကို သင်ကြားရန်အတွက်၊ ပထမ ဦး စွာ ကျွန်ုပ်တို့၏ ဒေတာစနစ်ကို GloVe အသုံးအနှုန်းစာရင်းဖြင့် ကုဒ်ပြုရမည်။


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

အထက်မှာ ကျွန်တော်တို့ မြင်ခဲ့သလို၊ vector embedding များအားလုံးကို `vocab.vectors` matrix တွင် သိမ်းဆည်းထားသည်။ ၎င်းသည် embedding အလွှာ၏ အလေးချိန်များထဲသို့ အလွယ်တကူ ကော်ပီလုပ်၍ ထည့်သွင်းခြင်းကို အလွန်လွယ်ကူစေသည်။


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

အခု မော်ဒယ်ကို လေ့ကျင့်ကြည့်ပြီး ရလဒ်က ပိုကောင်းလာမလား ကြည့်ကြမယ်။


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

ကျွန်ုပ်တို့အား သက်ဆိုင်ရာတိကျမှုမြင့်တက်မှု မထင်ရှားသေးခြင်း၏ အကြောင်းတစ်ခုမှာ ကျွန်ုပ်တို့၏ဒေတာစုဆောင်းမှုမှ စကားလုံးအချို့သည် ကြိုတင်လေ့လာထားသော GloVe အဘိဓါန်တွင် မပါဝင်သောကြောင့် အဓိပ္ပါယ်ဖျောက်ထားကြသောကြောင့် ဖြစ်သည်။ ဤအချက်ကို ဖြေရှင်းရန်အတွက် ကျွန်ုပ်တို့၏ဒေတာပေါ်တွင် ကိုယ်ပိုင် embeddings များကို လေ့ကျင့်နိုင်ပါသည်။


## စုံလင်သည့် သရုပ်ဖော်ချက်များ

Word2Vec ကဲ့သို့ မူလတန်းသင်ထားသော embedding ကိုယ်စားပြုချက်များ၏ အဓိက အကန့်အသတ်တစ်ခုမှာ စကားလုံး အဓိပ္ပာယ် ခြားနားတိက်ကျမွမ်းမံခြင်းပြဿနာဖြစ်သည်။ မူလတန်းသင်ထားသော embedding များသည် စကားလုံးများ၏ ဖော်ပြချက်အနည်းငယ်ကို သိရှိနိုင်သော်လည်း စကားလုံးတစ်လုံး၏ အဓိပ္ပာယ် များစွာကို တစ်ခုတည်းသော embedding ထဲတွင် ကောက်နှုတ်ထားသည်။ ၎င်းသည် နောက်ဆက်တွဲ မော်ဒယ်များတွင် ပြဿနာများ ဖြစ်ပေါ်စေနိုင်သည်၊ အကြောင်းက 'play' စကားလုံးကဲ့သို့ စကားလုံးများသည် သုံးသည့် သဘောတတ်ရှိမှု အပေါ် မူတည်၍ အဓိပ္ပာယ်ကွဲပြားမှုရှိသည်။

ဥပမာအားဖြင့် 'play' စကားလုံးသည် အောက်ပါ ဆက်ဆံစာကြောင်း နှစ်ခုတွင် အဓိပ္ပာယ် မတူကွဲပြားသည်။
- ကျွန်တော်က ပြဇာတ်ရုံတွင် **play** တစ်ခုသို့ သွားခဲ့တယ်။
- John သည် သူ့မိတ်ဆွေများနှင့် **play** ခြင်းကို ကြိုက်သည်။

အထက်ပါ မူလတန်းသင်ထားသော embedding များတွင် 'play' စကားလုံး၏ အဓိပ္ပာယ်နှစ်မျိုးလုံးကို တစ်ခုတည်းသော embedding ရှိသည်။ ဤအကန့်အသတ်ကို ကျော်လွှားရန်၊ သင့်အနေဖြင့် **ဘာသာစကား မော်ဒယ်**အကြောင်း ကို အခြေခံ၍ embedding များ ဖန်တီးရမည်ဖြစ်ပြီး၊ ၎င်းမှာ စာသားကြီး များစွာကို သင်ကြားပြီး စကားလုံးများကို ချိတ်ဆက်အသုံးပြုနိုင်မှုကို *သိ* ရသည်။ စုံလင်သည့် သရုပ်ဖော်ချက်များ ဆွေးနွေးခြင်းသည် ဤသင်ခန်းစာ၏ ဧရိယာအတွင်းမပါသောကြောင့်၊ နောက်ထပ်အပိုင်းတွင် ဘာသာစကား မော်ဒယ်များကို ပြောဆိုစဉ်တွင် ပြန်လည် ဆက်သွယ်ပါမည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
